# 03 · Silver

Bronze → cleaned, typed and deduplicated **Delta table** at `data/silver/enem_cleaned`.

1. Drop columns not used downstream (exam answers, answer keys, socioeconomic questionnaire).
2. Rename to English snake_case.
3. Cast types: scores → `double` (INEP uses `,` or `.` as decimal separator), codes → `int`.
4. Keep only the latest record per `registration_id`.
5. Write with a Delta **`MERGE`** (upsert): re-running never duplicates candidates.

In [ ]:
%run ./00_setup.ipynb

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql.functions import col, regexp_replace, row_number, trim

DROP_COLUMNS = (
    [f"TX_RESPOSTAS_{area}" for area in ("CN", "CH", "LC", "MT")]    # exam answers
    + [f"TX_GABARITO_{area}" for area in ("CN", "CH", "LC", "MT")]   # answer keys
    + [f"Q{i:03d}" for i in range(1, 26)]                            # questionnaire Q001-Q025
)

RENAME_COLUMNS = {
    "NU_INSCRICAO": "registration_id",
    "NU_ANO": "year",
    "TP_FAIXA_ETARIA": "age_group",
    "TP_SEXO": "gender",
    "TP_COR_RACA": "race",
    "TP_ST_CONCLUSAO": "high_school_status",
    "TP_ANO_CONCLUIU": "high_school_completion_year",
    "TP_ESCOLA": "school_type",
    "TP_ENSINO": "education_type",
    "IN_TREINEIRO": "is_trainee",
    "SG_UF_PROVA": "exam_state",
    "TP_PRESENCA_CN": "presence_natural_sciences",
    "TP_PRESENCA_CH": "presence_humanities",
    "TP_PRESENCA_LC": "presence_languages",
    "TP_PRESENCA_MT": "presence_math",
    "NU_NOTA_CN": "score_natural_sciences",
    "NU_NOTA_CH": "score_humanities",
    "NU_NOTA_LC": "score_languages",
    "NU_NOTA_MT": "score_math",
    "TP_LINGUA": "foreign_language",
    "TP_STATUS_REDACAO": "essay_status",
    "NU_NOTA_COMP1": "essay_comp1",
    "NU_NOTA_COMP2": "essay_comp2",
    "NU_NOTA_COMP3": "essay_comp3",
    "NU_NOTA_COMP4": "essay_comp4",
    "NU_NOTA_COMP5": "essay_comp5",
    "NU_NOTA_REDACAO": "score_essay",
}

# Types derived from the new names: text ids, decimal scores, everything else is an integer code
STRING_COLUMNS = {"registration_id", "gender", "exam_state"}
DOUBLE_COLUMNS = {c for c in RENAME_COLUMNS.values() if c.startswith(("score_", "essay_comp"))}
INT_COLUMNS = set(RENAME_COLUMNS.values()) - STRING_COLUMNS - DOUBLE_COLUMNS

In [ ]:
df = (
    spark.read.format("delta").load(BRONZE_TABLE)
    .drop(*DROP_COLUMNS)
    .withColumnsRenamed(RENAME_COLUMNS)
)

df = df.withColumns(
    {c: trim(col(c)) for c in STRING_COLUMNS}
    | {c: col(c).cast("int") for c in INT_COLUMNS}
    | {c: regexp_replace(col(c), ",", ".").cast("double") for c in DOUBLE_COLUMNS}
)

latest_first = Window.partitionBy("registration_id").orderBy(col("_ingestion_timestamp").desc())
df_silver = (
    df.filter(col("registration_id").isNotNull() & (col("registration_id") != ""))
    .withColumn("_row", row_number().over(latest_first))
    .filter(col("_row") == 1)
    .drop("_row")
)

Data quality checks before writing: the cell fails if a rule is broken.

In [ ]:
total = df_silver.count()
assert total > 0, "Silver is empty"
assert df_silver.select("registration_id").distinct().count() == total, "Duplicated registration_id"
assert df_silver.filter((col("score_math") < 0) | (col("score_math") > 1000)).isEmpty(), "Math score out of range"
print(f"Quality checks passed: {total} candidates")

In [ ]:
if DeltaTable.isDeltaTable(spark, SILVER_TABLE):
    (
        DeltaTable.forPath(spark, SILVER_TABLE).alias("t")
        .merge(df_silver.alias("s"), "t.registration_id = s.registration_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_silver.write.format("delta").partitionBy("year", "exam_state").save(SILVER_TABLE)

In [ ]:
silver = spark.read.format("delta").load(SILVER_TABLE)
print(f"Silver: {silver.count()} rows | {len(silver.columns)} columns")
silver.select("registration_id", "year", "exam_state", "gender", "score_math", "score_essay").show(5)

### Delta Lake: history and time travel
Run this notebook twice: the second run shows up as a `MERGE` version in the history.

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_TABLE}`").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

In [ ]:
# Read the table exactly as it was in its first version
silver_v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_TABLE)
print(f"Version 0: {silver_v0.count()} rows | current: {silver.count()} rows")

In [ ]:
spark.stop()